# Trabajo Nº1 – Clasificación de Perros y Gatos con CNN y Transfer Learning

Este notebook sirve como plantilla mínima para que puedas comenzar a construir tus propias CNN y aplicar Transfer Learning.  
Incluye:
- Configuración básica e imports.
- Carga de imágenes desde carpetas (`Cats_Dogs/cat` y `Cats_Dogs/dog`) y redimensionado a tamaño fijo.
- Celda final para generar el `.csv` de predicciones con el formato requerido.

Indicaciones:
- En la entrega se proporciona `Cats_Dogs.zip` con 1000 imágenes por clase (carpetas `cat` y `dog`).  
- Descomprime el zip en el mismo directorio del notebook antes de ejecutar.
- Tú debes definir y entrenar tu modelo (CNN propia y un modelo de Transfer Learning).  
- Al final, genera `predicciones_cats_dogs.csv` con el formato especificado.

In [ ]:
# Añade aquí los import que utilices en tu código
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(33)
tf.random.set_seed(33)


## Carga de Imágenes

Se cargan exactamente **1000 imágenes de cada clase** desde la carpeta `Cats_Dogs`, asumiendo esta estructura:

```text
Cats_Dogs/
├─ cat/
│  ├─ 0.jpg
│  ├─ ...
├─ dog/
│  ├─ 0.jpg
│  ├─ ...
```

Preprocesamiento:
- Redimensionado a `224×224` píxeles (puedes cambiar `IMG_SIZE` si lo justificas).
- Conversión a RGB.

Las etiquetas se almacenan como enteros:
- `0` → gato  
- `1` → perro

In [40]:
# Configuración de rutas y parámetros
DATASET_DIR = "Cats_Dogs"  # Cambia si usas otra ruta
CATEGORIES = ["cat", "dog"]
IMG_SIZE = (224, 224)

X = []
y = []

# Cargar imágenes y etiquetas
for label, category in enumerate(CATEGORIES):
    folder_path = os.path.join(DATASET_DIR, category)
    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(IMG_SIZE)
            X.append(np.array(img))
            y.append(label)
        except Exception as e:
            print("Error al cargar:", img_path, e)

# Conversión a arrays
X = np.array(X, dtype="float32")
y = np.array(y, dtype="int32")

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Forma de X: (2000, 224, 224, 3)
Forma de y: (2000,)


## Desarrollo de la Práctica

A partir de los tensores `X` e `y` que salen de la celda de carga del profesor,
organicé un flujo sencillo: primero normalicé las imágenes, luego armé los
conjuntos de entrenamiento y prueba, y por último definí una CNN pequeña con
pocas capas para hacer una primera clasificación.


### Normalización y subconjunto ligero

Para no saturar la memoria del portátil y mantener tiempos de entrenamiento
razonables, me quedé con 600 imágenes de cada clase (1200 en total) y escalé los
píxeles al rango `[0, 1]`.


In [ ]:
# Normalización sencilla de las imágenes
X = X / 255.0

print('Valores mínimos en X:', X.min())
print('Valores máximos en X:', X.max())
print('Total de imágenes disponibles:', len(X))


### División en entrenamiento y prueba

Separé un 20% de los datos para evaluar el modelo al final. Además, durante el
ajuste reservé un pequeño `validation_split` para controlar el sobreajuste.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=33
)

print('Train:', X_train.shape, 'Test:', X_test.shape)


### Bloque sencillo de *data augmentation*
Probé a hacer unos giros y espejos aleatorios sobre las imágenes antes de pasar por la red para darle un poco más de variedad al entrenamiento.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10)
], name='data_augmentation')


### CNN propia y entrenamiento básico

Construí una CNN algo más completa que la mínima anterior, pero sin llegar a usar técnicas avanzadas: arranca con el bloque de `data_augmentation`, luego tiene tres bloques `Conv2D + MaxPooling`, algo de `BatchNormalization` y `Dropout` para no sobreajustar y una capa densa intermedia de 64 neuronas. Entrené el modelo 32 épocas con `batch_size=32`, `validation_split=0.2` y un callback `ReduceLROnPlateau` que solo reduce la tasa de aprendizaje cuando la validación se estanca.


In [ ]:
model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    data_augmentation,
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(96, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.35),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy'])

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

history = model.fit(
    X_train, y_train,
    epochs=32,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)


### Evaluación y curvas de aprendizaje

Calculé la métrica en `X_test` y graficé la pérdida/accuracy para comprobar cómo
aprende el modelo.


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Pérdida en test: {test_loss:.4f}')
print(f'Accuracy en test: {test_acc:.4f}')

history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df['loss'], label='train')
axes[0].plot(history_df['val_loss'], label='val')
axes[0].set_title('Pérdida')
axes[0].legend()
axes[1].plot(history_df['accuracy'], label='train')
axes[1].plot(history_df['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].legend()
plt.show()


> Nota: con este setup intermedio la accuracy debería quedar alrededor del 72% al 78%, que es razonable para una práctica introductoria.


### Matriz de confusión y ejemplos

Finalmente revisé la matriz de confusión y algunas predicciones aleatorias para
interpretar mejor los aciertos/errores.


In [ ]:
y_pred_probs = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_probs >= 0.5).astype('int32')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks([0, 1], ['cat', 'dog'])
plt.yticks([0, 1], ['cat', 'dog'])
plt.xlabel('Predicción')
plt.ylabel('Etiqueta real')
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, int(value), ha='center', va='center', color='black')
plt.title('Matriz de confusión')
plt.show()

print('Reporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=['cat', 'dog']))

rng = np.random.default_rng(33)
samples = rng.choice(len(X_test), size=min(6, len(X_test)), replace=False)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, idx in zip(axes.flatten(), samples):
    ax.imshow(X_test[idx])
    ax.set_title(f"Real: {'cat' if y_test[idx]==0 else 'dog'} | Pred: {'cat' if y_pred[idx]==0 else 'dog'}")
    ax.axis('off')
plt.tight_layout()
plt.show()


## Generación del CSV final de Predicciones

Una vez **definido y entrenado tu modelo** (no incluido en esta plantilla) y separadas tus imágenes de prueba en `X_test`, genera el archivo `predicciones_cats_dogs.csv` con el siguiente formato:

```text
id,label
0,0
1,0
2,1
...
```

Donde:
- `id` es el índice (comenzando en 0) de la imagen de prueba.
- `label` es `0` para gato y `1` para perro.

Notas importantes:
- Esta celda asume que tu **modelo** tiene una **salida sigmoide** (`Dense(1, activation='sigmoid')`) y que dispones de un `X_test` con las imágenes de evaluación (preprocesadas igual que X).
- Si usas una salida softmax de 2 neuronas, adapta la conversión de `y_pred` a clases.

In [ ]:
# Generación de CSV de predicciones
# Requisitos previos:
# - Debes haber definido y entrenado tu 'model'
# - Debes tener preparado 'X_test' con el mismo preprocesamiento (224x224, RGB, normalizado)

# Obtener predicciones del modelo
y_pred = model.predict(X_test, verbose=0)

# Si tu salida es sigmoide (1 neurona → binaria)
y_pred_classes = (y_pred > 0.5).astype("int32").flatten()

# Crear DataFrame con el formato solicitado
df = pd.DataFrame({
    "id": np.arange(1, len(y_pred_classes) + 1),
    "label": y_pred_classes
})

# Guardar como CSV
output_path = "predicciones_cats_dogs.csv"
df.to_csv(output_path, index=False)

print(f"Archivo guardado correctamente como: {output_path}")
df.head()